# One-time step: build a slim MIMIC-IV input (≈1–2 GB instead of 40 GB)
Writes: icustays, admissions, patients, d_items; the vital-sign items for **every time point** of each ICU stay
(`chartevents_vitals_all.parquet`, with `hours_from_intime`) and the first-48 h subset used by the analysis
(`chartevents_vitals_48h.parquet`); laboratory results from 6 h before to 48 h after ICU admission
(`labs_48h.parquet`); ventilation episodes (`procedures_vent.parquet`); and vasopressor infusions
(`inputs_vaso.parquet`). To keep more chartevents items, add their itemids to `ITEMS`.
Runtime with the full dataset attached: roughly 60–90 min (labevents is the largest step).
Run this **once** with the full MIMIC-IV v3.1 dataset attached. Then:
1. **Save Version → Save & Run All** (outputs are kept only for saved versions).
2. Open the saved version → **Output** → **New Dataset** → name it `mimic-slim`, visibility **Private**.
3. In the analysis notebook, detach the full MIMIC-IV dataset and attach only `mimic-slim`.

Keep everything private: these files are MIMIC-IV data under the PhysioNet data use agreement.

In [ ]:
import os, glob, gc, time
import numpy as np, pandas as pd

DATA_ROOT = os.environ.get('DATA_ROOT', '/kaggle/input')
OUT_DIR   = os.environ.get('OUT_DIR', '/kaggle/working/mimic_slim')
os.makedirs(OUT_DIR, exist_ok=True)

def find(pattern):
    hits = sorted(glob.glob(os.path.join(DATA_ROOT, '**', pattern), recursive=True))
    if not hits:
        raise FileNotFoundError(f'{pattern} not found under {DATA_ROOT} — is the full MIMIC-IV dataset attached?')
    return hits[0]

PATH_ICU = find('icustays.csv*')
PATH_ADM = find('admissions.csv*')
PATH_CE  = find('chartevents.csv*')
def find_opt(pattern):
    hits = sorted(glob.glob(os.path.join(DATA_ROOT, '**', pattern), recursive=True))
    return hits[0] if hits else None
PATH_PAT = find_opt('patients.csv*')
PATH_DIT = find_opt('d_items.csv*')
PATH_LAB = find_opt('labevents.csv*')
PATH_PRO = find_opt('procedureevents.csv*')
PATH_INP = find_opt('inputevents.csv*')
print(PATH_ICU, PATH_ADM, PATH_CE, PATH_PAT, PATH_DIT, PATH_LAB, PATH_PRO, PATH_INP, sep='\n')

ITEMS = {
    220210: 'Respiratory rate', 220277: 'SpO2', 220045: 'Heart rate', 220179: 'NIBP systolic',
    223761: 'Temperature F', 223762: 'Temperature C', 220739: 'GCS eye', 223900: 'GCS verbal',
    226732: 'O2 delivery device', 223834: 'O2 flow (L/min)',
    220050: 'Arterial BP systolic', 220181: 'NIBP mean', 220052: 'Arterial BP mean', 223901: 'GCS motor',
}
LAB_ITEMS = {50813: 'Lactate', 50912: 'Creatinine', 51006: 'Urea nitrogen', 51301: 'WBC', 51300: 'WBC count',
             51265: 'Platelets', 50885: 'Bilirubin total', 51222: 'Hemoglobin', 50882: 'Bicarbonate',
             50971: 'Potassium', 50931: 'Glucose', 50983: 'Sodium'}
VENT_ITEMS = {225792: 'Invasive ventilation', 225794: 'Non-invasive ventilation'}
VASO_ITEMS = {221906: 'Norepinephrine', 221289: 'Epinephrine', 221749: 'Phenylephrine',
              222315: 'Vasopressin', 221662: 'Dopamine'}
WINDOW_H = 48   # analysis file keeps the first 48 h; the '_all' file keeps every time point


In [ ]:
icu = pd.read_csv(PATH_ICU, usecols=['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'intime', 'outtime', 'los'],
                  parse_dates=['intime', 'outtime'])
adm = pd.read_csv(PATH_ADM, usecols=['subject_id', 'hadm_id', 'admission_type', 'deathtime', 'hospital_expire_flag'],
                  parse_dates=['deathtime'])
icu.to_parquet(f'{OUT_DIR}/icustays.parquet', index=False)
adm.to_parquet(f'{OUT_DIR}/admissions.parquet', index=False)
print(f'icustays: {len(icu):,} rows | admissions: {len(adm):,} rows')
if PATH_PAT:
    pd.read_csv(PATH_PAT).to_parquet(f'{OUT_DIR}/patients.parquet', index=False); print('patients saved')
if PATH_DIT:
    pd.read_csv(PATH_DIT).to_parquet(f'{OUT_DIR}/d_items.parquet', index=False); print('d_items saved')
intime = icu.set_index('stay_id')['intime']


In [ ]:
t0 = time.time(); kept = []
reader = pd.read_csv(PATH_CE, chunksize=2_000_000,
                     usecols=['stay_id', 'itemid', 'charttime', 'value', 'valuenum'],
                     dtype={'value': 'string'}, low_memory=False,
                     keep_default_na=False, na_values=[''])   # keep the literal text 'None' (no O2 device)
for i, ch in enumerate(reader):
    ch = ch[ch['itemid'].isin(ITEMS.keys())]
    if ch.empty: continue
    ch = ch.dropna(subset=['stay_id'])
    ch['valuenum'] = pd.to_numeric(ch['valuenum'], errors='coerce')
    ch['charttime'] = pd.to_datetime(ch['charttime'], errors='coerce')
    ch['hours_from_intime'] = ((ch['charttime'] - ch['stay_id'].map(intime)).dt.total_seconds() / 3600).astype('float32')
    ch['value'] = ch['value'].where(ch['itemid'] == 226732)   # text kept only for the O2 device
    kept.append(ch)
    if i % 25 == 0:
        print(f'chunk {i:4d} | kept {sum(len(k) for k in kept):,} rows | {time.time()-t0:,.0f}s')
    gc.collect()

ce = pd.concat(kept, ignore_index=True); del kept; gc.collect()
ce['stay_id'] = ce['stay_id'].astype('int32')
ce['itemid'] = ce['itemid'].astype('int32')
ce['valuenum'] = ce['valuenum'].astype('float32')
ce['value'] = ce['value'].astype('category')
ce.to_parquet(f'{OUT_DIR}/chartevents_vitals_all.parquet', index=False)
print(f'All time points: {len(ce):,} rows for {ce.stay_id.nunique():,} stays')
ce48 = ce[(ce['hours_from_intime'] >= 0) & (ce['hours_from_intime'] < WINDOW_H)]
ce48.to_parquet(f'{OUT_DIR}/chartevents_vitals_48h.parquet', index=False)
print(f'First {WINDOW_H} h: {len(ce48):,} rows for {ce48.stay_id.nunique():,} stays | {time.time()-t0:,.0f}s')
print(ce['itemid'].map(ITEMS).value_counts().to_string())
print('O2 device values:', ce.loc[ce.itemid == 226732, 'value'].value_counts().head(15).to_dict())


In [ ]:
# Laboratory results, mapped to ICU stays by patient and time (−6 h to +48 h around ICU admission)
if PATH_LAB:
    t0, kept = time.time(), []
    st = icu[['subject_id', 'stay_id', 'intime']]
    for i, ch in enumerate(pd.read_csv(PATH_LAB, chunksize=3_000_000, usecols=['subject_id', 'itemid', 'charttime', 'valuenum'])):
        ch = ch[ch['itemid'].isin(LAB_ITEMS.keys())].dropna(subset=['valuenum'])
        if ch.empty: continue
        ch['charttime'] = pd.to_datetime(ch['charttime'], errors='coerce')
        ch = ch.merge(st, on='subject_id', how='inner')
        ch['hours_from_intime'] = ((ch['charttime'] - ch['intime']).dt.total_seconds() / 3600).astype('float32')
        ch = ch[(ch['hours_from_intime'] >= -6) & (ch['hours_from_intime'] < WINDOW_H)]
        kept.append(ch[['stay_id', 'itemid', 'hours_from_intime', 'valuenum']])
        if i % 10 == 0: print(f'lab chunk {i:4d} | kept {sum(len(k) for k in kept):,} | {time.time()-t0:,.0f}s')
        gc.collect()
    lab = pd.concat(kept, ignore_index=True); del kept
    lab = lab.astype({'stay_id': 'int32', 'itemid': 'int32', 'valuenum': 'float32'})
    lab.to_parquet(f'{OUT_DIR}/labs_48h.parquet', index=False)
    print(f'labs: {len(lab):,} rows, {lab.stay_id.nunique():,} stays | {time.time()-t0:,.0f}s')
    print(lab['itemid'].map(LAB_ITEMS).value_counts().to_string())
else:
    print('⚠ labevents not found — lab features will be skipped')


In [ ]:
# Ventilation episodes (procedureevents) and vasopressor infusions (inputevents)
def episodes(path, items, out, label):
    if not path: print(f'⚠ {label} table not found — skipped'); return
    kept = []
    for ch in pd.read_csv(path, chunksize=2_000_000, usecols=['stay_id', 'itemid', 'starttime', 'endtime']):
        ch = ch[ch['itemid'].isin(items.keys())].dropna(subset=['stay_id'])
        if len(ch): kept.append(ch)
    e = pd.concat(kept, ignore_index=True)
    e = e.merge(icu[['stay_id', 'intime']], on='stay_id', how='inner')
    for c in ['starttime', 'endtime']:
        e[f'{c[:-4]}_h'] = ((pd.to_datetime(e[c], errors='coerce') - e['intime']).dt.total_seconds() / 3600).astype('float32')
    e = e[(e['start_h'] < WINDOW_H) & (e['end_h'] >= -6)][['stay_id', 'itemid', 'start_h', 'end_h']]
    e.astype({'stay_id': 'int32', 'itemid': 'int32'}).to_parquet(f'{OUT_DIR}/{out}', index=False)
    print(f'{label}: {len(e):,} episodes in {e.stay_id.nunique():,} stays'); print(e['itemid'].map(items).value_counts().to_string())
episodes(PATH_PRO, VENT_ITEMS, 'procedures_vent.parquet', 'ventilation')
episodes(PATH_INP, VASO_ITEMS, 'inputs_vaso.parquet', 'vasopressors')


In [ ]:
for f in sorted(os.listdir(OUT_DIR)):
    print(f'{f:40s} {os.path.getsize(os.path.join(OUT_DIR, f))/1e6:8.1f} MB')
